In [ ]:
TRAIN_IMG_DIR   = r'Dataset\sets\raw_labeled_data\images'
TRAIN_MASK_DIR  = r'Dataset\sets\raw_labeled_data\annotations'

TEST_AGANA_DIR  = r'Dataset\sets\test_data\1\images'
TEST_AGANA_GT   = r'Dataset\sets\test_data\1\annotations'
TEST_MANELL_DIR = r'Dataset\sets\test_data\3\images'
TEST_MANELL_GT  = r'Dataset\sets\test_data\3\annotations'

OUTPUT_DIR = './output_masks'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Dataset constants
NUM_BANDS= 8  
NUM_CLASSES= 7  
PATCH_SIZE= 256
STRIDE= 128

# ****** students can modify the training hyperparameters below ******
BATCH_SIZE = 8
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
VAL_SPLIT     = 0.2  # fraction of labeled images held out for validation
SEED          = 42

# Class names and RGB display colors (matches the assignment legend)
CLASS_NAMES  = ['Background','Seagrass','Coral','Macroalgae','Sand','Land','Ocean']
CLASS_COLORS = [
    [30,  30,  30],   # Background
    [34, 139,  34],   # Seagrass
    [255,140,   0],   # Coral
    [0,  100,   0],   # Macroalgae
    [210,180, 140],   # Sand
    [139, 90,  43],   # Land
    [135,206, 235],   # Ocean
]

In [ ]:
def compute_band_stats(img_dir):
    paths = sorted(glob.glob(os.path.join(img_dir, '*.tif')))
    assert len(paths) > 0, f'No .tif files found in {img_dir}'
    running_sum = np.zeros(NUM_BANDS, dtype=np.float64)
    running_sum2 = np.zeros(NUM_BANDS, dtype=np.float64)
    pixel_count = 0
    for p in paths:
        with rasterio.open(p) as src:
            data = src.read().astype(np.float64)  # (8, H, W)
        n = data.shape[1] * data.shape[2]
        running_sum += data.sum(axis=(1, 2))
        running_sum2 += (data ** 2).sum(axis=(1, 2))
        pixel_count += n
    mean = running_sum / pixel_count
    std  = np.sqrt(running_sum2 / pixel_count - mean ** 2)
    std  = np.where(std < 1e-8, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


print('Computing normalisation statistics...')
BAND_MEAN, BAND_STD = compute_band_stats(TRAIN_IMG_DIR)
print('Mean per band:', np.round(BAND_MEAN, 1))
print('Std  per band:', np.round(BAND_STD,  1))

In [ ]:

class BenthicDataset(Dataset):
    def __init__(self, img_paths, mask_paths, patch_size, stride,
                 band_mean, band_std, augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.patch_size = patch_size
        self.stride = stride
        self.mean = band_mean[:, None, None] 
        self.std = band_std[:, None, None]
        self.augment = augment

        self.patch_index = []
        for i, p in enumerate(img_paths):
            with rasterio.open(p) as src:
                H, W = src.height, src.width
            for top  in range(0, H - patch_size + 1, stride):
                for left in range(0, W - patch_size + 1, stride):
                    self.patch_index.append((i, top, left))

    def __len__(self):
        return len(self.patch_index)

    def __getitem__(self, idx):
        img_idx, top, left = self.patch_index[idx]
        ps     = self.patch_size
        window = rasterio.windows.Window(left, top, ps, ps)


        with rasterio.open(self.img_paths[img_idx]) as src:
            img = src.read(window=window).astype(np.float32)  # (8, ps, ps)
        img = (img - self.mean) / self.std
        img = torch.from_numpy(img)

        # Load mask patch
        if self.mask_paths is not None:
            with rasterio.open(self.mask_paths[img_idx]) as src:
                mask = src.read(1, window=window).astype(np.int64)  # (ps, ps)
            mask = torch.from_numpy(mask)
        else:
            mask = torch.zeros(ps, ps, dtype=torch.int64)
        if self.augment:
            if random.random() > 0.5:  
                img  = torch.flip(img,  dims=[2])
                mask = torch.flip(mask, dims=[1])
            if random.random() > 0.5: 
                img  = torch.flip(img,  dims=[1])
                mask = torch.flip(mask, dims=[0])
            k = random.randint(0, 3) 
            if k > 0:
                img  = torch.rot90(img,  k, dims=[1, 2])
                mask = torch.rot90(mask, k, dims=[0, 1])

        return img, mask

In [ ]:
train_imgs = sorted(glob.glob(os.path.join(TRAIN_IMG_DIR,  '*.tif')))
train_masks = sorted(glob.glob(os.path.join(TRAIN_MASK_DIR, '*.tif')))
assert len(train_imgs) == len(train_masks), 'Image/mask count mismatch'

# Shuffle deterministically so the val split isn't a contiguous alphabetical slice
rng = random.Random(SEED)
pairs = list(zip(train_imgs, train_masks))
rng.shuffle(pairs)
train_imgs, train_masks = zip(*pairs)
train_imgs, train_masks = list(train_imgs), list(train_masks)

n_val = max(1, int(len(train_imgs) * VAL_SPLIT))
val_imgs,  val_masks  = train_imgs[:n_val],  train_masks[:n_val]
fit_imgs,  fit_masks  = train_imgs[n_val:],  train_masks[n_val:]

train_ds = BenthicDataset(fit_imgs, fit_masks, PATCH_SIZE, STRIDE,
                           BAND_MEAN, BAND_STD, augment=True)
val_ds = BenthicDataset(val_imgs,  val_masks,  PATCH_SIZE, PATCH_SIZE,
                           BAND_MEAN, BAND_STD, augment=False)

trainloader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
valloader = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Training patches:   {len(train_ds):,}')
print(f'Validation patches: {len(val_ds):,}')

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_rate=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),    
            nn.ReLU(inplace=True),             
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate),    
        )
    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x):
        return self.pool_conv(x)


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
      
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        diffY = skip.size(2) - x.size(2)
        diffX = skip.size(3) - x.size(3)
        x = F.pad(x, [diffX//2, diffX - diffX//2, diffY//2, diffY - diffY//2])
        x = torch.cat([skip, x], dim=1)  # skip connection: concat along channel dim
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=8, num_classes=7, base_filters=64):
        super(UNet, self).__init__()
        f = base_filters
        # Encoder
        self.enc1 = DoubleConv(in_channels, f)   # 256x256  ch: 8  -> 64
        self.enc2 = Down(f,f*2)              # 128x128  ch: 64 -> 128
        self.enc3 = Down(f*2,f*4)              #  64x64   ch:128 -> 256
        self.enc4 = Down(f*4,f*8)              #  32x32   ch:256 -> 512
        # Bottleneck
        self.bottleneck = Down(f*8,  f*16)             #  16x16   ch:512 -> 1024
        # Decoder
        self.dec4  = Up(f*16, f*8, f*8)          #  32x32
        self.dec3 = Up(f*8, f*4, f*4)          #  64x64
        self.dec2 = Up(f*4, f*2, f*2)          # 128x128
        self.dec1 = Up(f*2, f, f)            # 256x256
        self.out_conv   = nn.Conv2d(f, num_classes, 1)

    def forward(self, x):
        # Encoder — save each level for the skip connections
        s1 = self.enc1(x)  
        s2 = self.enc2(s1)     
        s3 = self.enc3(s2)   
        s4 = self.enc4(s3)     
        x  = self.bottleneck(s4)  
        # Decoder — each Up block receives the skip from the matching encoder level
        x  = self.dec4(x, s4)   
        x  = self.dec3(x, s3)   
        x  = self.dec2(x, s2) 
        x  = self.dec1(x, s1) 
        return self.out_conv(x) 


net = UNet(in_channels=NUM_BANDS, num_classes=NUM_CLASSES)
net = nn.DataParallel(net)
net.to(device)
print(f'Total parameters: {sum(p.numel() for p in net.parameters() if p.requires_grad):,}')

In [ ]:

CLASS_FREQ = np.array([0.0350, 0.0306, 0.0765, 0.1191, 0.0860, 0.3789, 0.2741],
                       dtype=np.float32)

class_weights  = 1.0 / (CLASS_FREQ + 1e-6)
class_weights /= class_weights.mean()
class_weights_t = torch.tensor(class_weights, device=device)

print('Class weights (higher = rarer class):')
for name, w in zip(CLASS_NAMES, class_weights):
    print(f'  {name:12s}: {w:.2f}')


class DiceLoss(nn.Module):

    def __init__(self, num_classes, smooth=1.0):
        super().__init__()
        self.C = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)            
        targets_oh = F.one_hot(targets, self.C)     
        targets_oh = targets_oh.permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)  # sum over batch, H, W; keep class axis
        inter = (probs * targets_oh).sum(dims)
        card   = (probs + targets_oh).sum(dims)
        return 1.0 - ((2 * inter + self.smooth) / (card + self.smooth)).mean()


class CombinedLoss(nn.Module):

    def __init__(self, weights, num_classes):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss(weight=weights)
        self.dice = DiceLoss(num_classes)

    def forward(self, logits, targets):
        return 0.5 * self.ce(logits, targets) + 0.5 * self.dice(logits, targets)


loss_fn = CombinedLoss(class_weights_t, NUM_CLASSES)

In [ ]:
opt = optim.AdamW(net.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.5, patience=5
)


In [ ]:
def iou_from_confusion(conf):
    # Per-class IoU + mean over classes that appear at least once in GT or pred
    ious = []
    for c in range(conf.shape[0]):
        tp = conf[c, c]
        denom = conf[:, c].sum() + conf[c, :].sum() - tp
        if denom > 0:
            ious.append(tp / denom)
    return float(np.mean(ious)) if ious else 0.0


def test(net, dataloader):
    test_loss   = 0.0
    test_correct = 0
    num_pixels = 0
    conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    net.eval()
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = net(inputs)                          # (B, 7, H, W) logits
            test_loss += loss_fn(outputs, labels).item() * inputs.size(0)
            preds = torch.max(outputs, dim=1)[1]
            test_correct += (preds == labels).sum().item()
            num_pixels   += labels.numel()

            # Accumulate confusion matrix across the whole validation set
            p = preds.view(-1).cpu().numpy()
            t = labels.view(-1).cpu().numpy()
            valid = (t >= 0) & (t < NUM_CLASSES)
            np.add.at(conf, (t[valid], p[valid]), 1)

    return (test_loss / len(dataloader.dataset),
            test_correct / num_pixels * 100,
            iou_from_confusion(conf) * 100)

In [ ]:
# These lists store loss and accuracy values for plotting
train_los = []
train_cor = []
test_los = []
test_cor = []
test_miou_history = []

# Track best model performance
net_corr, net_los, net_epoch = 0, float('inf'), 0

for epoch in range(NUM_EPOCHS):
    loss_avg  = 0
    correct = 0
    num_pixels = 0
    net.train()

    for inputs, labels in trainloader:
        inputs = inputs.to(device)  
        labels = labels.to(device)  

        outputs = net(inputs)
        loss = loss_fn(outputs, labels)

        opt.zero_grad()   
        loss.backward()  
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0) 
        opt.step()                    

        preds = torch.max(outputs.data, dim=1)[1]
        correct    += (preds == labels).sum().item()
        num_pixels += labels.numel()
        loss_avg   += loss.item() * inputs.size(0)

    # Evaluate on validation set
    los2, cor2, miou2 = test(net, valloader)
    scheduler.step(los2)

    tr_acc = correct / num_pixels * 100
    tr_los = loss_avg / len(train_ds)

    print('Training: {}/{} epoch, LR: {:.6f}, Train Loss: {:.4f}, Train Acc: {:.2f}%, '
          'Val Loss: {:.4f}, Val Acc: {:.2f}%, Val mIoU: {:.2f}%'
          .format(epoch+1, NUM_EPOCHS,
                  opt.state_dict()['param_groups'][0]['lr'],
                  tr_los, tr_acc, los2, cor2, miou2))

    train_los.append(tr_los)
    train_cor.append(tr_acc)
    test_los.append(los2)
    test_cor.append(cor2)
    test_miou_history.append(miou2)

    if los2 < net_los:
        net_corr, net_los, net_epoch = cor2, los2, epoch + 1
        # Save the underlying module's state_dict (portable across DataParallel / single-GPU)
        state = net.module.state_dict() if isinstance(net, nn.DataParallel) else net.state_dict()
        torch.save(state, './best_unet.pt')

print('The {} epoch achieves the best model, Val Loss: {:.4f}, Val Acc: {:.2f}%'
      .format(net_epoch, net_los, net_corr))

In [ ]:
# Loss curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(range(NUM_EPOCHS), train_los, label='Train')
plt.plot(range(NUM_EPOCHS), test_los,  label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss')
plt.legend(); plt.xticks(range(0, NUM_EPOCHS, max(1, NUM_EPOCHS//10)))

# Accuracy curves
plt.subplot(1, 3, 2)
plt.plot(range(NUM_EPOCHS), train_cor, label='Train')
plt.plot(range(NUM_EPOCHS), test_cor,  label='Val')
plt.xlabel('Epoch'); plt.ylabel('Pixel Accuracy (%)'); plt.title('Accuracy')
plt.legend(); plt.xticks(range(0, NUM_EPOCHS, max(1, NUM_EPOCHS//10)))

# Validation mIoU
plt.subplot(1, 3, 3)
plt.plot(range(NUM_EPOCHS), test_miou_history)
plt.xlabel('Epoch'); plt.ylabel('mIoU (%)'); plt.title('Validation mIoU')
plt.xticks(range(0, NUM_EPOCHS, max(1, NUM_EPOCHS//10)))

plt.tight_layout()
plt.savefig('./training_curves.png', dpi=150)
plt.show()

In [ ]:
from torchinfo import summary

summary(UNet(in_channels=NUM_BANDS, num_classes=NUM_CLASSES).to(device),
        input_size=(BATCH_SIZE, NUM_BANDS, PATCH_SIZE, PATCH_SIZE))

In [ ]:
# Rebuild a fresh model and load the saved weights (portable, no pickle issues)
net = UNet(in_channels=NUM_BANDS, num_classes=NUM_CLASSES).to(device)
net.load_state_dict(torch.load('./best_unet.pt', map_location=device))
net.eval()


def _patch_offsets(dim, patch, stride):
    # Offsets that cover [0, dim) including the last edge (clamped to dim - patch)
    if dim <= patch:
        return [0]
    offs = list(range(0, dim - patch + 1, stride))
    if offs[-1] != dim - patch:
        offs.append(dim - patch)
    return offs


def predict_full_image(img_path):
    mean = BAND_MEAN[:, None, None]
    std  = BAND_STD[:, None, None]
    with rasterio.open(img_path) as src:
        img = src.read().astype(np.float32)   # (8, H, W)
    _, H, W   = img.shape

    # Pad if image is smaller than a patch in either dimension
    pad_h = max(0, PATCH_SIZE - H)
    pad_w = max(0, PATCH_SIZE - W)
    if pad_h or pad_w:
        img = np.pad(img, ((0, 0), (0, pad_h), (0, pad_w)), mode='reflect')

    _, Hp, Wp = img.shape
    img_norm  = (img - mean) / std
    logit_acc = np.zeros((NUM_CLASSES, Hp, Wp), dtype=np.float32)
    count_acc = np.zeros((1, Hp, Wp), dtype=np.float32)

    with torch.no_grad():
        for top in _patch_offsets(Hp, PATCH_SIZE, STRIDE):
            for left in _patch_offsets(Wp, PATCH_SIZE, STRIDE):
                patch  = img_norm[:, top:top+PATCH_SIZE, left:left+PATCH_SIZE]
                t = torch.from_numpy(patch).unsqueeze(0).to(device)
                logits = net(t).squeeze(0).cpu().numpy()           # (7, ps, ps)
                logit_acc[:, top:top+PATCH_SIZE, left:left+PATCH_SIZE] += logits
                count_acc[:, top:top+PATCH_SIZE, left:left+PATCH_SIZE] += 1

    count_acc = np.where(count_acc == 0, 1, count_acc)
    pred = (logit_acc / count_acc).argmax(axis=0).astype(np.uint8)
    # Crop back to original size if padded
    return pred[:H, :W]


def mask_to_color(mask):
    # Convert integer class mask to RGB for visualisation
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for c, color in enumerate(CLASS_COLORS):
        rgb[mask == c] = color
    return rgb

In [ ]:
# Compare predictions against ground truth and save error visualisations

import imageio

def _read_gt(gt_path):
    # Use rasterio for .tif (multiband-safe) and imageio for .png
    if gt_path.lower().endswith('.tif'):
        with rasterio.open(gt_path) as src:
            gt = src.read(1)
    else:
        gt = np.array(imageio.imread(gt_path))
        if gt.ndim == 3:
            gt = gt[:, :, 0]
    return gt.astype(np.int64)


def evaluate_and_save(img_dir, gt_dir, region_name, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    img_paths = sorted(glob.glob(os.path.join(img_dir, '*.tif')))
    all_ious  = []

    for img_path in img_paths:
        stem  = os.path.splitext(os.path.basename(img_path))[0]
        pred  = predict_full_image(img_path)
        color = mask_to_color(pred)

        # Submission: single-channel class-index mask (0..NUM_CLASSES-1)
        imageio.imwrite(os.path.join(save_dir, f'{stem}_pred.png'),
                        pred.astype(np.uint8))
        # Also save a coloured version for human inspection
        plt.imsave(os.path.join(save_dir, f'{stem}_pred_color.png'), color)

        # Load ground truth
        gt_path = os.path.join(gt_dir, f'{stem}.png')
        if not os.path.exists(gt_path):
            gt_path = os.path.join(gt_dir, f'{stem}.tif')
        if not os.path.exists(gt_path):
            continue

        gt = _read_gt(gt_path)

        # Per-image IoU via confusion matrix (no per-batch averaging bias)
        conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
        valid = (gt >= 0) & (gt < NUM_CLASSES)
        np.add.at(conf, (gt[valid], pred[valid]), 1)
        iou = iou_from_confusion(conf)
        all_ious.append(iou)

        # Error map: pixels where prediction != ground truth shown in red
        error_map = color.copy()
        error_map[pred != gt] = [220, 20, 60]

        # RGB display using bands 3-2-1 (Green/Blue/Coastal) with percentile stretch
        with rasterio.open(img_path) as src:
            rgb = src.read([3, 2, 1]).transpose(1, 2, 0).astype(np.float32)
        for ch in range(3):
            lo, hi = np.percentile(rgb[:,:,ch], [2, 98])
            rgb[:,:,ch] = np.clip((rgb[:,:,ch]-lo)/(hi-lo+1e-6)*255, 0, 255)
        rgb = rgb.astype(np.uint8)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(rgb); axes[0].set_title('Input (RGB composite)'); axes[0].axis('off')
        axes[1].imshow(color); axes[1].set_title(f'Prediction  mIoU={iou*100:.1f}%'); axes[1].axis('off')
        axes[2].imshow(error_map); axes[2].set_title('Error map (red = wrong pixels)'); axes[2].axis('off')

        patches = [mpatches.Patch(color=[c/255 for c in col], label=nm)
                   for nm, col in zip(CLASS_NAMES, CLASS_COLORS)]
        fig.legend(handles=patches, loc='lower center', ncol=7, fontsize=8)
        plt.suptitle(f'{region_name} - {stem}')
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f'{stem}_analysis.png'), dpi=120)
        plt.close()

    if all_ious:
        print(f'{region_name}: mean mIoU = {np.mean(all_ious)*100:.2f}%  (n={len(all_ious)} images)')
    return all_ious


agana_ious  = evaluate_and_save(TEST_AGANA_DIR,  TEST_AGANA_GT,
                                 'Agana Bay',   os.path.join(OUTPUT_DIR, 'agana'))
manell_ious = evaluate_and_save(TEST_MANELL_DIR, TEST_MANELL_GT,
                                 'Manell-Geus', os.path.join(OUTPUT_DIR, 'manell'))

print(f'Overall mIoU: {np.mean(agana_ious + manell_ious)*100:.2f}%')

In [ ]:
# Per-class IoU breakdown using an accumulated confusion matrix

def per_class_iou(img_dir, gt_dir, region_name):
    conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for img_path in sorted(glob.glob(os.path.join(img_dir, '*.tif'))):
        stem    = os.path.splitext(os.path.basename(img_path))[0]
        gt_path = os.path.join(gt_dir, f'{stem}.png')
        if not os.path.exists(gt_path): continue
        pred = predict_full_image(img_path)
        gt   = np.array(imageio.imread(gt_path))
        if gt.ndim == 3: gt = gt[:,:,0]
        valid = (gt >= 0) & (gt < NUM_CLASSES)
        np.add.at(conf, (gt[valid], pred[valid]), 1)

    print(f'\n=== {region_name} per-class IoU ===')
    ious = []
    for c in range(NUM_CLASSES):
        tp  = conf[c, c]
        denom = conf[:,c].sum() + conf[c,:].sum() - tp
        iou = tp / denom if denom > 0 else 0.0
        ious.append(iou)
        print(f'  {CLASS_NAMES[c]:12s}: {iou*100:.1f}%')
    print(f'  {"mIoU":12s}: {np.mean(ious)*100:.1f}%')
    return ious


per_class_iou(TEST_AGANA_DIR,  TEST_AGANA_GT,  'Agana Bay')
per_class_iou(TEST_MANELL_DIR, TEST_MANELL_GT, 'Manell-Geus')

In [ ]:
with zipfile.ZipFile('./submission_masks.zip', 'w') as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files:
            if f.endswith('_pred.png'):
                fp = os.path.join(root, f)
                zf.write(fp, os.path.relpath(fp, OUTPUT_DIR))

print('Saved submission_masks.zip')